# Model

In [1]:
import datetime
import dateutil.relativedelta as rd
import pandas as pd
import statsmodels.formula.api as smf
from tqdm import tqdm

## Universe

In [2]:
stocks = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'BRK-B', 'JNJ', 'JPM', 'XOM', 'BAC',
          'V', 'PFE', 'PG', 'CVX', 'HD', 'VZ', 'MA', 'WMT', 'CSCO', 'INTC',
          'KO', 'DIS', 'PEP', 'MRK', 'ABT', 'NKE', 'T', 'CMCSA', 'ORCL', 'IBM',
          'C', 'WFC', 'PM', 'GS', 'NFLX', 'BA', 'MCD', 'MMM', 'HON', 'AXP',
          'AVGO', 'UNP', 'UPS', 'ABBV', 'CAT', 'MS', 'CVS', 'UNH', 'LMT', 'TGT']

In [3]:
comps = {'AAPL':  ['AAPL', 'MSFT', 'GOOGL', 'IBM', 'ORCL'],
         'MSFT':  ['MSFT', 'AAPL', 'GOOGL', 'IBM', 'ORCL'],
         'GOOGL': ['GOOGL', 'AAPL', 'MSFT', 'NFLX', 'META'],
         'AMZN':  ['AMZN', 'HD', 'TGT', 'WMT', 'DIS'],
         'JNJ':   ['JNJ', 'PFE', 'MRK', 'ABT', 'ABBV'],
         'JPM':   ['JPM', 'BAC', 'C', 'WFC', 'GS'],
         'XOM':   ['XOM', 'CVX', 'COP', 'PSX', 'OXY'],
         'BAC':   ['BAC', 'JPM', 'C', 'WFC', 'MS'],
         'V':     ['V', 'MA', 'AXP', 'C', 'BAC'],
         'PFE':   ['PFE', 'JNJ', 'MRK', 'ABT', 'ABBV'],
         'PG':    ['PG', 'KO', 'PEP', 'WMT', 'PM'],
         'CVX':   ['CVX', 'XOM', 'COP', 'PSX', 'OXY'],
         'HD':    ['HD', 'LOW', 'TGT', 'WMT', 'AMZN'],
         'VZ':    ['VZ', 'T', 'TMUS', 'CMCSA', 'AMZN'],
         'MA':    ['MA', 'V', 'AXP', 'C', 'BAC'],
         'WMT':   ['WMT', 'TGT', 'COST', 'AMZN', 'HD'],
         'CSCO':  ['CSCO', 'INTC', 'IBM', 'ORCL', 'AVGO'],
         'INTC':  ['INTC', 'CSCO', 'IBM', 'ORCL', 'AVGO'],
         'KO':    ['KO', 'PEP', 'MNST', 'PG', 'PM'],
         'DIS':   ['DIS', 'CMCSA', 'NFLX', 'T', 'AMZN'],
         'PEP':   ['PEP', 'KO', 'PG', 'WMT', 'PM'],
         'MRK':   ['MRK', 'JNJ', 'PFE', 'ABT', 'ABBV'],
         'ABT':   ['ABT', 'JNJ', 'PFE', 'MRK', 'ABBV'],
         'NKE':   ['NKE', 'ADDYY', 'UAA', 'MCD', 'HD'],
         'T':     ['T', 'VZ', 'TMUS', 'CMCSA', 'DIS'],
         'CMCSA': ['CMCSA', 'DIS', 'NFLX', 'T', 'VZ'],
         'ORCL':  ['ORCL', 'IBM', 'MSFT', 'CSCO', 'INTC'],
         'IBM':   ['IBM', 'ORCL', 'MSFT', 'CSCO', 'INTC'],
         'C':     ['C', 'JPM', 'BAC', 'WFC', 'GS'],
         'WFC':   ['WFC', 'JPM', 'BAC', 'C', 'MS'],
         'PM':    ['PM', 'MO', 'KO', 'PEP', 'PG'],
         'GS':    ['GS', 'JPM', 'C', 'BAC', 'MS'],
         'NFLX':  ['NFLX', 'GOOGL', 'AMZN', 'DIS', 'META'],
         'BA':    ['BA', 'LMT', 'HON', 'MMM', 'CAT'],
         'MCD':   ['MCD', 'YUM', 'SBUX', 'NKE', 'HD'],
         'MMM':   ['MMM', 'HON', 'BA', 'CAT', 'GE'],
         'HON':   ['HON', 'MMM', 'BA', 'CAT', 'LMT'],
         'AXP':   ['AXP', 'V', 'MA', 'BAC', 'JPM'],
         'AVGO':  ['AVGO', 'CSCO', 'INTC', 'IBM', 'ORCL'],
         'UNP':   ['UNP', 'CSX', 'NSC', 'UPS', 'BA'],
         'UPS':   ['UPS', 'FDX', 'UNP', 'BA', 'CAT'],
         'ABBV':  ['ABBV', 'JNJ', 'PFE', 'MRK', 'ABT'],
         'CAT':   ['CAT', 'DE', 'BA', 'MMM', 'HON'],
         'MS':    ['MS', 'GS', 'JPM', 'BAC', 'WFC'],
         'CVS':   ['CVS', 'WBA', 'UNH', 'JNJ', 'PFE'],
         'UNH':   ['UNH', 'HUM', 'CVS', 'JNJ', 'ELV'],
         'LMT':   ['LMT', 'BA', 'GD', 'NOC', 'HON'],
         'TGT':   ['TGT', 'WMT', 'AMZN', 'COST', 'HD'],
         'META':  ['GOOGL', 'AAPL', 'MSFT', 'NFLX', 'META'],
         'BRK-B': ['JPM', 'BAC', 'C', 'GS', 'BRK-B']}

## Prices

In [4]:
df_prices = pd.read_csv('data/prices.csv', index_col='date', parse_dates=True)

In [5]:
for ticker in stocks + ['SPY']:
    df_prices[f'{ticker}_return'] = (df_prices[f'{ticker}_price']
                                             - df_prices[f'{ticker}_price'].shift(1))/df_prices[f'{ticker}_price'].shift(1)

df_prices

C:\Users\notep\AppData\Local\Temp\ipykernel_15520\3742676923.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prices[f'{ticker}_return'] = (df_prices[f'{ticker}_price']


,AAPL_price,MSFT_price,GOOGL_price,AMZN_price,META_price,BRK-B_price,JNJ_price,JPM_price,XOM_price,BAC_price,...,UNP_return,UPS_return,ABBV_return,CAT_return,MS_return,CVS_return,UNH_return,LMT_return,TGT_return,SPY_return
date,,,,,,,,,,,,,,,,,,,,,
2015-12-01,26.384714,48.046402,38.820091,33.952999,106.103760,136.429993,76.370476,50.612267,51.804806,14.039582,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-12-02,26.146362,48.037701,38.525902,33.800499,105.063728,135.270004,76.139191,49.901115,50.324486,13.928911,...,-0.027488,-0.010347,-0.022027,-0.007407,-0.010774,-0.009953,0.016241,-0.005722,-0.012086,-0.010205
2015-12-03,25.903515,47.158905,38.047951,33.312500,103.389755,132.660004,75.199089,49.257324,49.603313,13.675947,...,-0.035088,-0.009777,-0.027720,-0.019569,-0.012611,-0.017460,-0.022441,-0.015895,0.017517,-0.014003
2015-12-04,26.764713,48.646763,38.593258,33.632000,105.172691,136.479996,76.810669,50.821869,49.887978,14.071203,...,-0.011237,0.009483,0.018888,0.004882,0.025254,0.028110,0.023826,0.013599,0.009701,0.019503
2015-12-07,26.596069,48.559765,38.285194,33.491501,104.608093,135.600006,77.086723,50.155624,48.584797,13.865671,...,-0.013025,-0.007845,-0.015914,-0.023149,-0.018120,0.009219,-0.015543,0.003388,-0.006901,-0.006058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-24,68.402298,148.487335,66.587860,89.460503,203.174042,225.470001,121.305222,114.395409,51.953457,29.918219,...,0.002620,-0.005129,-0.004432,-0.006869,-0.000592,0.001748,-0.001864,-0.004414,0.001636,0.000031
2019-12-26,69.759422,149.704407,67.481369,93.438499,205.818726,226.449997,121.222137,115.609383,52.035069,30.173061,...,0.005114,0.006000,-0.000222,0.005018,0.008286,-0.000403,0.003769,0.005459,0.002645,0.005323
2019-12-27,69.732941,149.978073,67.093559,93.489998,206.125793,226.139999,121.155617,115.692528,51.857018,30.028643,...,0.003319,-0.000840,-0.007014,0.000405,-0.000391,-0.001074,0.001082,0.003314,0.002483,-0.000247


## Wikipedia pageviews

In [6]:
df_views = pd.read_csv('data/views.csv', index_col='date', parse_dates=True)

In [7]:
for ticker in stocks:
    df_views[f'{ticker}_prop_views'] = df_views[f'{ticker}_views'] / df_views[[f'{t}_views' for t in comps[ticker]]].sum(axis=1)

df_views

C:\Users\notep\AppData\Local\Temp\ipykernel_15520\1821364318.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_views[f'{ticker}_prop_views'] = df_views[f'{ticker}_views'] / df_views[[f'{t}_views' for t in comps[ticker]]].sum(axis=1)
C:\Users\notep\AppData\Local\Temp\ipykernel_15520\1821364318.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_views[f'{ticker}_prop_views'] = df_views[f'{ticker}_views'] / df_views[[f'{t}_views' for t in comps[ticker]]].sum(axis=1)
C:\Users\notep\AppData\Local\Temp\ipykernel_15520\18213643

,AAPL_views,MSFT_views,GOOGL_views,IBM_views,ORCL_views,NFLX_views,META_views,AMZN_views,HD_views,TGT_views,...,AVGO_prop_views,UNP_prop_views,UPS_prop_views,ABBV_prop_views,CAT_prop_views,MS_prop_views,CVS_prop_views,UNH_prop_views,LMT_prop_views,TGT_prop_views
date,,,,,,,,,,,,,,,,,,,,,
2015-12-01,16070.0,9372.0,6040.0,6059.0,3947.0,11055.0,104.0,14806.0,1730.0,3825.0,...,0.069899,0.168981,0.197342,0.054630,0.184903,0.152197,0.046307,0.270565,0.264398,0.115674
2015-12-02,16713.0,10063.0,6572.0,6158.0,4310.0,11054.0,158.0,13740.0,1752.0,3308.0,...,0.071569,0.193068,0.200418,0.059643,0.193228,0.128336,0.051918,0.307438,0.289170,0.104833
2015-12-03,16173.0,9371.0,5957.0,6232.0,3941.0,10506.0,114.0,13183.0,1481.0,3078.0,...,0.075293,0.190771,0.189537,0.076280,0.210235,0.132354,0.066961,0.307944,0.293525,0.103539
2015-12-04,17248.0,13383.0,5548.0,5468.0,3658.0,10053.0,128.0,13141.0,1648.0,2690.0,...,0.077198,0.181248,0.198281,0.078176,0.218618,0.124269,0.073045,0.321898,0.294413,0.091781
2015-12-05,11864.0,6699.0,3853.0,4157.0,2548.0,10044.0,78.0,11622.0,1272.0,2428.0,...,0.069127,0.189254,0.192941,0.066990,0.185497,0.113920,0.077156,0.268357,0.276409,0.097710
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-27,11789.0,7577.0,6782.0,4271.0,2901.0,18856.0,5503.0,15580.0,2406.0,2804.0,...,0.082214,0.128565,0.147879,0.103499,0.144253,0.118369,0.099297,0.212044,0.221630,0.091004
2019-12-28,11499.0,6874.0,6125.0,4611.0,2711.0,18286.0,5682.0,14381.0,2297.0,2675.0,...,0.062672,0.142190,0.168086,0.096633,0.146856,0.159516,0.115188,0.206381,0.269138,0.092776
2019-12-29,11292.0,7997.0,5859.0,4582.0,2767.0,18579.0,5840.0,15908.0,2094.0,2630.0,...,0.057045,0.166554,0.166049,0.097957,0.172970,0.119369,0.122163,0.232846,0.260139,0.086459


In [8]:
df_data = pd.merge(df_prices,
                   df_views,
                   left_index=True,
                   right_index=True,
                   how='left')
df_data

,AAPL_price,MSFT_price,GOOGL_price,AMZN_price,META_price,BRK-B_price,JNJ_price,JPM_price,XOM_price,BAC_price,...,AVGO_prop_views,UNP_prop_views,UPS_prop_views,ABBV_prop_views,CAT_prop_views,MS_prop_views,CVS_prop_views,UNH_prop_views,LMT_prop_views,TGT_prop_views
date,,,,,,,,,,,,,,,,,,,,,
2015-12-01,26.384714,48.046402,38.820091,33.952999,106.103760,136.429993,76.370476,50.612267,51.804806,14.039582,...,0.069899,0.168981,0.197342,0.054630,0.184903,0.152197,0.046307,0.270565,0.264398,0.115674
2015-12-02,26.146362,48.037701,38.525902,33.800499,105.063728,135.270004,76.139191,49.901115,50.324486,13.928911,...,0.071569,0.193068,0.200418,0.059643,0.193228,0.128336,0.051918,0.307438,0.289170,0.104833
2015-12-03,25.903515,47.158905,38.047951,33.312500,103.389755,132.660004,75.199089,49.257324,49.603313,13.675947,...,0.075293,0.190771,0.189537,0.076280,0.210235,0.132354,0.066961,0.307944,0.293525,0.103539
2015-12-04,26.764713,48.646763,38.593258,33.632000,105.172691,136.479996,76.810669,50.821869,49.887978,14.071203,...,0.077198,0.181248,0.198281,0.078176,0.218618,0.124269,0.073045,0.321898,0.294413,0.091781
2015-12-07,26.596069,48.559765,38.285194,33.491501,104.608093,135.600006,77.086723,50.155624,48.584797,13.865671,...,0.066762,0.195271,0.183484,0.083148,0.180120,0.119102,0.080060,0.192790,0.263261,0.103182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-24,68.402298,148.487335,66.587860,89.460503,203.174042,225.470001,121.305222,114.395409,51.953457,29.918219,...,0.081628,0.057105,0.085497,0.145830,0.106159,0.124796,0.082805,0.195415,0.136017,0.100854
2019-12-26,69.759422,149.704407,67.481369,93.438499,205.818726,226.449997,121.222137,115.609383,52.035069,30.173061,...,0.087722,0.152868,0.133614,0.108253,0.142775,0.119634,0.100202,0.211145,0.202754,0.087818
2019-12-27,69.732941,149.978073,67.093559,93.489998,206.125793,226.139999,121.155617,115.692528,51.857018,30.028643,...,0.082214,0.128565,0.147879,0.103499,0.144253,0.118369,0.099297,0.212044,0.221630,0.091004


## Walk-forward

In [9]:
train_start    = datetime.datetime(2016, 1, 1)
train_length   = rd.relativedelta(years=1)
test_length    = rd.relativedelta(months=1)

n_cycles       = 36

In [10]:
coefficients = []
for ticker in tqdm(stocks):
    df_data['this_return']       = df_data[f'{ticker}_return']
    df_data['lagged_return']     = df_data[f'{ticker}_return'].shift(1).rolling(window=7).mean()
    df_data['lagged_views']      = df_data[f'{ticker}_views'].shift(1).rolling(window=7).mean()
    df_data['lagged_prop_views'] = df_data[f'{ticker}_prop_views'].shift(1).rolling(window=7).mean()

    df_data[f'{ticker}_pred_1'] = 0.0
    df_data[f'{ticker}_pred_2'] = 0.0
    df_data[f'{ticker}_pred_3'] = 0.0

    for cycle_n in range(n_cycles):
        train_index = (df_data.index >= (train_start + (cycle_n * test_length)))                & (df_data.index < (train_start + (cycle_n * test_length) + train_length              ))
        test_index  = (df_data.index >= (train_start + (cycle_n * test_length) + train_length)) & (df_data.index < (train_start + (cycle_n * test_length) + train_length + test_length))

        model_1 = smf.ols('this_return ~ lagged_return', data=df_data[train_index]).fit()
        model_2 = smf.ols('this_return ~ lagged_return + lagged_views', data=df_data[train_index]).fit()
        model_3 = smf.ols('this_return ~ lagged_return + lagged_views + lagged_prop_views', data=df_data[train_index]).fit()

        df_data.loc[test_index, f'{ticker}_pred_1'] = model_1.predict(df_data[test_index])
        df_data.loc[test_index, f'{ticker}_pred_2'] = model_2.predict(df_data[test_index])
        df_data.loc[test_index, f'{ticker}_pred_3'] = model_3.predict(df_data[test_index])

        for model_id, model in [(1, model_1), (2, model_2), (3, model_3)]:
            conf_int = model.conf_int()
            for term in model.params.index:
                coefficients.append({'ticker'  : ticker,
                                     'cycle'   : cycle_n + 1,
                                     'model'   : model_id,
                                     'term'    : term,
                                     'coef'    : model.params[term],
                                     'std_err' : model.bse[term],
                                     'ci_low'  : conf_int.loc[term, 0],
                                     'ci_high' : conf_int.loc[term, 1]})

    df_data = df_data.copy()

  0%|          | 0/50 [00:00<?, ?it/s]

C:\Users\notep\AppData\Local\Temp\ipykernel_15520\1984294540.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_data['this_return']       = df_data[f'{ticker}_return']
C:\Users\notep\AppData\Local\Temp\ipykernel_15520\1984294540.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_data['lagged_return']     = df_data[f'{ticker}_return'].shift(1).rolling(window=7).mean()
C:\Users\notep\AppData\Local\Temp\ipykernel_15520\1984294540.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

  2%|▏         | 1/50 [00:01<01:36,  1.97s/it]

  4%|▍         | 2/50 [00:03<01:14,  1.56s/it]

  6%|▌         | 3/50 [00:04<01:04,  1.38s/it]

...

 94%|█████████▍| 47/50 [00:53<00:03,  1.15s/it]

 96%|█████████▌| 48/50 [00:54<00:02,  1.12s/it]

 98%|█████████▊| 49/50 [00:56<00:01,  1.13s/it]

100%|██████████| 50/50 [00:57<00:00,  1.11s/it]

100%|██████████| 50/50 [00:57<00:00,  1.14s/it]

In [11]:
full_test_index = (df_data.index >= train_start + train_length) & (df_data.index < train_start + train_length + n_cycles*test_length)

df_test = df_data[full_test_index].copy()
df_test

,AAPL_price,MSFT_price,GOOGL_price,AMZN_price,META_price,BRK-B_price,JNJ_price,JPM_price,XOM_price,BAC_price,...,CVS_pred_3,UNH_pred_1,UNH_pred_2,UNH_pred_3,LMT_pred_1,LMT_pred_2,LMT_pred_3,TGT_pred_1,TGT_pred_2,TGT_pred_3
date,,,,,,,,,,,,,,,,,,,,,
2017-01-03,26.698210,55.944389,40.019688,37.683498,115.751381,163.830002,88.870537,67.225090,59.529247,18.089121,...,-0.001731,0.001600,0.000942,-0.001071,0.001012,0.002065,0.004894,-0.001788,-0.000659,-0.000629
2017-01-04,26.668329,55.694077,40.007797,37.859001,117.563995,164.080002,88.724770,67.349075,58.874283,18.426338,...,-0.002177,0.001470,0.000817,-0.001138,0.000751,0.001717,0.004147,-0.001409,-0.000194,-0.000164
2017-01-05,26.803944,55.694077,40.267826,39.022499,119.525230,163.300003,89.653084,66.729134,57.996643,18.209549,...,-0.002260,0.001454,0.000924,-0.000879,0.000729,0.001536,0.003711,0.000187,0.001062,0.001087
2017-01-06,27.102762,56.176826,40.871578,39.799500,122.239220,163.410004,89.223457,66.736870,57.963875,18.209549,...,-0.002188,0.001545,0.001324,-0.000031,0.000436,0.000143,0.001093,-0.000186,0.000363,0.000411
2017-01-09,27.351011,55.998024,40.969151,39.846001,123.715088,162.020004,89.208099,66.783340,57.007648,18.105177,...,-0.002252,0.001498,0.001432,0.000282,0.000404,0.000033,0.000878,-0.000681,-0.000177,-0.000123
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2019-12-24,68.402298,148.487335,66.587860,89.460503,203.174042,225.470001,121.305222,114.395409,51.953457,29.918219,...,-0.001825,0.000319,-0.000037,-0.000314,0.001180,0.001572,0.001257,0.002466,0.008893,0.008715
2019-12-26,69.759422,149.704407,67.481369,93.438499,205.818726,226.449997,121.222137,115.609383,52.035069,30.173061,...,-0.001558,0.000286,-0.000183,-0.000332,0.001228,0.001619,0.001171,0.002518,0.009354,0.009361
2019-12-27,69.732941,149.978073,67.093559,93.489998,206.125793,226.139999,121.155617,115.692528,51.857018,30.028643,...,-0.001720,0.000210,-0.000394,-0.000528,0.001197,0.001663,0.001206,0.002683,0.009193,0.009277


## Trading with our models

In [12]:
for strat_id in [1,2,3]:
    df_test[f'strat_return_{strat_id}'] = 0.0

    for row_n, row in df_test.iterrows():
        returns = []
        for ticker in stocks:
            returns.append( [row[f'{ticker}_pred_{strat_id}'], row[f'{ticker}_return']] )

        returns = sorted(returns, key = lambda i : i[0])
        returns = [i[1] for i in returns]

        df_test.loc[row_n, f'strat_return_{strat_id}'] = 1 + sum(returns[-10:])/10

## 2017 check

In [13]:
df_2017 = df_test[df_test.index < '2018-01-01']

pd.DataFrame({'cumulative_return' : [(1 + df_2017.SPY_return).prod(), df_2017.strat_return_1.prod(), df_2017.strat_return_2.prod(), df_2017.strat_return_3.prod()],
              'risk'              : [df_2017.SPY_return.std(), df_2017.strat_return_1.std(), df_2017.strat_return_2.std(), df_2017.strat_return_3.std()]},
             index = ['SPY', 'Strategy 1', 'Strategy 2', 'Strategy 3'])

,cumulative_return,risk
SPY,1.217054,0.004248
Strategy 1,1.332950,0.007098
Strategy 2,1.431013,0.006061
Strategy 3,1.405994,0.006178


## Save results

In [14]:
df_strategy_returns = df_test[['strat_return_1', 'strat_return_2', 'strat_return_3']] - 1
df_strategy_returns.index.name = 'date'
df_strategy_returns.to_csv('data/strategy_returns.csv')

In [15]:
df_predictions = df_test[[f'{ticker}_pred_{strat_id}' for strat_id in [1,2,3] for ticker in stocks]]
df_predictions.index.name = 'date'
df_predictions.to_csv('data/predictions.csv')

In [16]:
df_coefficients = pd.DataFrame(coefficients)
df_coefficients.to_csv('data/coefficients.csv', index=False)
df_coefficients

,ticker,cycle,model,term,coef,std_err,ci_low,ci_high
0,AAPL,1,1,Intercept,5.014194e-04,9.304544e-04,-1.331109e-03,2.333948e-03
1,AAPL,1,1,lagged_return,1.414902e-01,1.636563e-01,-1.808306e-01,4.638109e-01
2,AAPL,1,2,Intercept,-1.559219e-03,3.688090e-03,-8.823049e-03,5.704610e-03
3,AAPL,1,2,lagged_return,1.333737e-01,1.644765e-01,-1.905689e-01,4.573163e-01
4,AAPL,1,2,lagged_views,1.282197e-07,2.220417e-07,-3.090996e-07,5.655391e-07
...,...,...,...,...,...,...,...,...
16195,TGT,36,2,lagged_views,5.298391e-06,2.686110e-06,7.788449e-09,1.058899e-05
16196,TGT,36,3,Intercept,-1.461059e-02,1.331240e-02,-4.083141e-02,1.161022e-02
16197,TGT,36,3,lagged_return,5.330427e-02,1.610224e-01,-2.638542e-01,3.704627e-01
16198,TGT,36,3,lagged_views,4.421546e-06,4.653572e-06,-4.744381e-06,1.358747e-05
